In [1]:
from pathlib import Path
import re
import pymol
import numpy as np

from rdkit import Chem


In [2]:

# ============================================================
# CONFIG
# ============================================================

REFERENCE_COMPLEX = Path(
    "./reference_structures/SARS-CoV-2-Mpro/complex.pdb"
)

PROJECT_ROOT = Path(".")

REFERENCE_CHAIN = "A"
TEMPLATE_CHAIN = "A"

OUTPUT_SDF = "best_sars.sdf"


# ============================================================
# METADATA PARSER
# ============================================================

def parse_metadata(metadata_path):

    data = {}

    with open(metadata_path, "r") as f:

        for line in f:

            line = line.strip()

            if line.startswith("Target ligand:"):
                data["target_ligand"] = line.split(":", 1)[1].strip()

            elif line.startswith("Template ligand:"):
                data["template_ligand"] = line.split(":", 1)[1].strip()

            elif line.startswith("Receptor:"):

                full_path = line.split(":", 1)[1].strip()

                marker = "receptors/"

                idx = full_path.find(marker)

                if idx == -1:
                    raise ValueError(
                        f"Could not find '{marker}' in:\n{full_path}"
                    )

                relative_part = full_path[idx:]

                data["receptor_relpath"] = Path("./") / relative_part

    return data


# ============================================================
# TRANSFORM APPLICATION
# ============================================================

def apply_pymol_transform(mol, matrix):

    conf = mol.GetConformer()

    transform = np.array(matrix).reshape(4, 4)

    for i in range(mol.GetNumAtoms()):

        pos = conf.GetAtomPosition(i)

        vec = np.array([pos.x, pos.y, pos.z, 1.0])

        new = transform @ vec

        conf.SetAtomPosition(
            i,
            (float(new[0]), float(new[1]), float(new[2]))
        )


# ============================================================
# ALIGN SINGLE LIGAND
# ============================================================

def align_ligand(ligand_folder, ligand_filename="best_pose.sdf"):

    ligand_folder = Path(ligand_folder)

    metadata_path = ligand_folder / "metadata.txt"

    if not metadata_path.exists():
        print(f"Skipping {ligand_folder} (no metadata)")
        return None

    metadata = parse_metadata(metadata_path)

    receptor_path = PROJECT_ROOT / metadata["receptor_relpath"]

    ligand_path = ligand_folder / ligand_filename

    if not ligand_path.exists():
        print(f"Missing ligand: {ligand_path}")
        return None

    mol = Chem.MolFromMolFile(str(ligand_path), removeHs=False)

    if mol is None:
        print(f"Failed to read ligand: {ligand_path}")
        return None

    print("\n===================================================")
    print(f"Processing {ligand_folder.name}")
    print(f"Target ligand    : {metadata['target_ligand']}")
    print(f"Template ligand  : {metadata['template_ligand']}")
    print(f"Template receptor: {receptor_path}")
    print("===================================================")

    pymol.cmd.reinitialize()

    pymol.cmd.load(str(REFERENCE_COMPLEX), "reference")
    pymol.cmd.load(str(receptor_path), "template")

    mobile_sel = f"template and chain {TEMPLATE_CHAIN} and name CA"
    target_sel = f"reference and chain {REFERENCE_CHAIN} and name CA"

    rms_before = pymol.cmd.rms_cur(mobile_sel, target_sel)
    print(f"RMSD before alignment: {rms_before:.3f} Å")

    pymol.cmd.align(mobile_sel, target_sel, quiet=0)

    rms_after = pymol.cmd.rms_cur(mobile_sel, target_sel)
    print(f"RMSD after alignment : {rms_after:.3f} Å")

    matrix = pymol.cmd.get_object_matrix("template")

    if matrix is None:
        print("No transform matrix")
        return None

    apply_pymol_transform(mol, matrix)

    mol.SetProp("_Name", metadata["target_ligand"])
    mol.SetProp("template_ligand", metadata["template_ligand"])
    mol.SetProp("template_receptor", str(receptor_path))

    return mol


# ============================================================
# NATURAL SORT (FIX FOR ligand_1, ligand_10 ISSUE)
# ============================================================

def get_ligand_index(path):
    match = re.search(r"ligand_(\d+)", path.name)
    return int(match.group(1)) if match else float("inf")


# ============================================================
# PROCESS ALL LIGANDS
# ============================================================

outputs_dir = Path("outputs")

ligand_folders = sorted(
    [f for f in outputs_dir.glob("ligand_*") if f.is_dir()],
    key=get_ligand_index
)

print(f"Found {len(ligand_folders)} ligand folders")

writer = Chem.SDWriter(OUTPUT_SDF)

success = 0
failed = 0

for folder in ligand_folders:

    try:
        mol = align_ligand(folder)

        if mol is not None:
            writer.write(mol)
            success += 1
        else:
            failed += 1

    except Exception as e:
        print(f"\nERROR in {folder}")
        print(e)
        failed += 1

writer.close()

print("\n======================================")
print("Finished")
print(f"Successful: {success}")
print(f"Failed    : {failed}")
print("======================================")

print(f"\nOutput written to: {OUTPUT_SDF}")

Found 93 ligand folders

Processing ligand_0
Target ligand    : ligand_0
Template ligand  : mol-218
Template receptor: receptors/receptor-218.pdb
RMSD before alignment: 41.026 Å
 Match: read scoring matrix.
 Match: assigning 305 x 306 pairwise scores.
 MatchAlign: aligning residues (305 vs 306)...
 MatchAlign: score 1641.000
 ExecutiveAlign: 305 atoms aligned.
 ExecutiveRMS: 14 atoms rejected during cycle 1 (RMSD=0.23).
 ExecutiveRMS: 8 atoms rejected during cycle 2 (RMSD=0.18).
 ExecutiveRMS: 4 atoms rejected during cycle 3 (RMSD=0.17).
 ExecutiveRMS: 2 atoms rejected during cycle 4 (RMSD=0.17).
 ExecutiveRMS: 2 atoms rejected during cycle 5 (RMSD=0.16).
 Executive: RMSD =    0.162 (275 to 275 atoms)
RMSD after alignment : 0.241 Å

Processing ligand_1
Target ligand    : ligand_1
Template ligand  : mol-218
Template receptor: receptors/receptor-218.pdb
RMSD before alignment: 41.026 Å
 Match: read scoring matrix.
 Match: assigning 305 x 306 pairwise scores.
 MatchAlign: aligning residues